In [4]:
import numpy as np
import pandas as pd
import time
import re
import html
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.dates as mdates
from deep_translator import GoogleTranslator
from bs4 import BeautifulSoup, Comment, MarkupResemblesLocatorWarning
from scipy.stats import mannwhitneyu
import warnings
import pickle
from tqdm import tqdm
from tqdm.notebook import tqdm
from collections import Counter
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import ComplementNB
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import IsolationForest
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import classification_report, accuracy_score, average_precision_score, f1_score, precision_recall_curve
from sklearn.metrics import precision_recall_fscore_support
from xgboost import XGBClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack

In [5]:
import os
import psutil
import platform

print("CPU cores:", os.cpu_count())
print("RAM available:",
      round(psutil.virtual_memory().available / (1024**3), 2), "GB")
print("RAM total:",
      round(psutil.virtual_memory().total / (1024**3), 2), "GB")
print("Processor:", platform.processor())

CPU cores: 12
RAM available: 1.48 GB
RAM total: 15.51 GB
Processor: ARMv8 (64-bit) Family 8 Model 2 Revision 201, Qualcomm Technologies Inc


### Load Dataset (with English translation)

In [6]:
# Load the clean dataset
pd.set_option('display.max_columns', None)
clean_data = pd.read_csv("all_data_with_english_translations.csv", usecols=['created_at', 'subdomain','table','label','is_spam','url_count','has_url','body_clean','language','body_english'])
clean_data['has_url'] = clean_data['has_url'].astype(int)
display(len(clean_data))
display(clean_data.head())

93323

,created_at,subdomain,table,is_spam,label,url_count,has_url,body_clean,language,body_english
0,2023-02-24 01:52:08,support,hc_answers,0,NaN,0,0,"Hi Serge, thanks for the feedback! Agreed that...",en,"Hi Serge, thanks for the feedback! Agreed that..."
1,2022-06-22 20:38:15,support,hc_comments,0,NaN,0,0,1263082275569 is there a way to remove the del...,en,1263082275569 is there a way to remove the del...
2,2022-11-04 14:14:18,support,hc_comments,0,NaN,0,0,Has anybody managed to get this setup using Az...,en,Has anybody managed to get this setup using Az...
3,2023-04-11 07:27:27,support,hc_answers,0,NaN,0,0,"Hi Rey,\n \nI created a ticket on your behalf ...",en,"Hi Rey,\n \nI created a ticket on your behalf ..."
4,2022-05-09 16:19:22,support,hc_answers,0,NaN,0,0,I too am trying to accomplish this but haven't...,en,I too am trying to accomplish this but haven't...


In [7]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option("display.max_colwidth", None)
display(len(clean_data[clean_data['label'] == 14]))

6

In [8]:
clean_data = clean_data[clean_data['label'] != 14].copy()

### Extract Features (Word Count)

In [9]:
#Add word count

clean_data['word_count'] = clean_data['body_english'].str.split().str.len()
display(clean_data.head())

,created_at,subdomain,table,is_spam,label,url_count,has_url,body_clean,language,body_english,word_count
0,2023-02-24 01:52:08,support,hc_answers,0,NaN,0,0,"Hi Serge, thanks for the feedback! Agreed that the side conversation notifications need to live in a place like the Home Dashboard like you suggested. Unfortunately, the existing dashboard activity stream is only designed to list specific types of Ticket events, but we do have other avenues we are currently exploring to make it easier to stay on top of the side conversation (and other) activities.",en,"Hi Serge, thanks for the feedback! Agreed that the side conversation notifications need to live in a place like the Home Dashboard like you suggested. Unfortunately, the existing dashboard activity stream is only designed to list specific types of Ticket events, but we do have other avenues we are currently exploring to make it easier to stay on top of the side conversation (and other) activities.",66.0
1,2022-06-22 20:38:15,support,hc_comments,0,NaN,0,0,1263082275569 is there a way to remove the deleted organizations from showing up in explorer? I use a report to export data and we have 500+ deleted accounts from our initial seeding that I would like to remove.,en,1263082275569 is there a way to remove the deleted organizations from showing up in explorer? I use a report to export data and we have 500+ deleted accounts from our initial seeding that I would like to remove.,38.0
2,2022-11-04 14:14:18,support,hc_comments,0,NaN,0,0,Has anybody managed to get this setup using Azure AD to create agents with a custom role?\nWe have added mappings for role (set to 'agent') and custom_role_id (set to the id of the custom role to assign) but users are always created as end-user rather than as agent.\n\nAny ideas?\nThanks,en,Has anybody managed to get this setup using Azure AD to create agents with a custom role?\nWe have added mappings for role (set to 'agent') and custom_role_id (set to the id of the custom role to assign) but users are always created as end-user rather than as agent.\n\nAny ideas?\nThanks,52.0
3,2023-04-11 07:27:27,support,hc_answers,0,NaN,0,0,"Hi Rey,\n \nI created a ticket on your behalf and will send it to you via email so we can further discuss your concern regarding your report.\n \nThank you!",en,"Hi Rey,\n \nI created a ticket on your behalf and will send it to you via email so we can further discuss your concern regarding your report.\n \nThank you!",29.0
4,2022-05-09 16:19:22,support,hc_answers,0,NaN,0,0,I too am trying to accomplish this but haven't been able to figure it out yet. If the ticket cannot be submitted by the end user can the form filler at least be CC'd on the ticket?,en,I too am trying to accomplish this but haven't been able to figure it out yet. If the ticket cannot be submitted by the end user can the form filler at least be CC'd on the ticket?,37.0


### Model Building

In [10]:
# 1. Define the exact columns you want to select
selected_cols = ['is_spam', 'url_count', 'has_url', 'word_count']

# Subset the DataFrame to only these columns
clean_data_filtered = clean_data[selected_cols]

# 2. Separate into features (X) and target prediction (y)
X = clean_data_filtered.drop(columns=['is_spam'])  # Features for the model
y = clean_data_filtered['is_spam']                 # What you want to predict

print("--- Features (X) ---")
display(X.head())

print("\n--- Target / Prediction (y) ---")
display(y.head())

print(len(X))
print(len(y))

--- Features (X) ---


,url_count,has_url,word_count
0,0,0,66.0
1,0,0,38.0
2,0,0,52.0
3,0,0,29.0
4,0,0,37.0



--- Target / Prediction (y) ---


0    0
1    0
2    0
3    0
4    0
Name: is_spam, dtype: int64

93317
93317


### Individual Features

In [11]:

# Features and target
features = ['url_count', 'has_url', 'word_count']

X = clean_data[features].fillna(0)
y = clean_data['is_spam']

# ==========================================
# 1. DYNAMIC INDIVIDUAL FEATURE EVALUATION
# ==========================================

feature_results = []

for feature in features:

    X_single = X[[feature]]

    # Split data for evaluation
    X_train, X_test, y_train, y_test = train_test_split(
        X_single,
        y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )

    # Calculate expected proportion of normal comments
    contamination = (y_train == 1).mean()

    # Initialize Isolation Forest
    model = IsolationForest(
        n_estimators=100,
        contamination=contamination,
        random_state=42
    )

    # Train without using labels
    model.fit(X_train)

    # Predict anomalies: -1 = anomaly, 1 = normal
    predictions = model.predict(X_test)

    # Convert predictions to spam labels:
    # -1 (anomaly) -> 1 (spam)
    #  1 (normal)  -> 0 (not spam)
    y_pred = (predictions == -1).astype(int)

    # Calculate spam-class performance
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_test,
        y_pred,
        labels=[0, 1],
        zero_division=0
    )

    feature_results.append({
        'Feature': feature,
        'Spam Precision': precision[1],
        'Spam Recall': recall[1],
        'Spam F1-Score': f1[1]
    })

# Results
df_features = pd.DataFrame(feature_results)

print("--- Individual Feature Performance (Isolation Forest) ---")

display(
    df_features.style.format({
        'Spam Precision': '{:.4f}',
        'Spam Recall': '{:.4f}',
        'Spam F1-Score': '{:.4f}'
    }).hide(axis='index')
)

--- Individual Feature Performance (Isolation Forest) ---


Feature,Spam Precision,Spam Recall,Spam F1-Score
url_count,0.1992,0.1153,0.1461
has_url,0.0000,0.0000,0.0000
word_count,0.3257,0.3253,0.3255


### Features (URL Count, Has URL, Word Count)

In [12]:

import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.ensemble import IsolationForest
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
    accuracy_score
)

# ==========================================
# 1. SELECT REQUIRED COLUMNS
# ==========================================

selected_cols = [
    'is_spam',
    'url_count',
    'has_url',
    'word_count'
]

clean_data_filtered = clean_data[selected_cols].copy()

# Separate features and target
X = clean_data_filtered.drop(columns=['is_spam']).fillna(0)
y = clean_data_filtered['is_spam']

# ==========================================
# 2. SPLIT TRAINING AND TESTING DATA
# ==========================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Training samples: {len(X_train)}")
print(f"Testing samples: {len(X_test)}")

print(f"Non-spam count: {(y_train == 0).sum()}")
print(f"Spam count: {(y_train == 1).sum()}")

# ==========================================
# 3. INITIALIZE ISOLATION FOREST
# ==========================================

model = IsolationForest(
    n_estimators=100,
    contamination=0.05,
    max_samples='auto',
    random_state=42
)

# ==========================================
# 4. TRAIN THE MODEL
# ==========================================

# Unsupervised training: no target labels used
model.fit(X_train)

# ==========================================
# 5. PREDICT ANOMALIES
# ==========================================

# Isolation Forest:
# -1 = Anomaly
#  1 = Normal

predictions = model.predict(X_test)

# Convert anomaly predictions into spam labels
#  1 = Spam
#  0 = Non-spam

y_pred = (predictions == -1).astype(int)

# ==========================================
# 6. MODEL EVALUATION
# ==========================================

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred,
        target_names=['Non-spam', 'Spam'],
        zero_division=0
    )
)

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nAccuracy Score:")
print(f"{accuracy_score(y_test, y_pred):.4f}")

# ==========================================
# 7. SPAM PERFORMANCE METRICS
# ==========================================

precision, recall, f1, _ = precision_recall_fscore_support(
    y_test,
    y_pred,
    labels=[0, 1],
    zero_division=0
)

results = pd.DataFrame([{
    'Algorithm': 'Isolation Forest',
    'Spam Precision': precision[1],
    'Spam Recall': recall[1],
    'Spam F1-Score': f1[1]
}])

print("\nCombined Feature Performance:")

display(
    results.style.format({
        'Spam Precision': '{:.4f}',
        'Spam Recall': '{:.4f}',
        'Spam F1-Score': '{:.4f}'
    }).hide(axis='index')
)


Training samples: 74653
Testing samples: 18664
Non-spam count: 71147
Spam count: 3506

Classification Report:
              precision    recall  f1-score   support

    Non-spam       0.97      0.97      0.97     17788
        Spam       0.35      0.36      0.35       876

    accuracy                           0.94     18664
   macro avg       0.66      0.66      0.66     18664
weighted avg       0.94      0.94      0.94     18664


Confusion Matrix:
[[17205   583]
 [  565   311]]

Accuracy Score:
0.9385

Combined Feature Performance:


Algorithm,Spam Precision,Spam Recall,Spam F1-Score
Isolation Forest,0.3479,0.3550,0.3514


### Only TF-IDF Features

In [15]:
from sklearn.cluster import DBSCAN
from sklearn.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score
)
import numpy as np
import pandas as pd

# ==========================================
# 1. SELECT SAMPLE
# ==========================================

sample_size = 5000

rng = np.random.RandomState(42)

indices = rng.choice(
    X_train_text.shape[0],
    size=sample_size,
    replace=False
)

X_sample = X_train_text[indices]
y_sample = y_train.iloc[indices]

# ==========================================
# 2. DBSCAN
# ==========================================

dbscan = DBSCAN(
    eps=0.5,
    min_samples=5,
    metric='cosine',
    n_jobs=-1
)

clusters = dbscan.fit_predict(X_sample)

# ==========================================
# 3. EVALUATION
# ==========================================

print("Number of clusters:",
      len(set(clusters)) - (1 if -1 in clusters else 0))

print("Noise points:",
      np.sum(clusters == -1))

print("ARI:",
      adjusted_rand_score(y_sample, clusters))

print("NMI:",
      normalized_mutual_info_score(y_sample, clusters))

# ==========================================
# 4. CLUSTER DISTRIBUTION
# ==========================================

results = pd.DataFrame({
    'Actual Spam': y_sample.values,
    'Cluster': clusters
})

display(pd.crosstab(
    results['Cluster'],
    results['Actual Spam']
))

Number of clusters: 32
Noise points: 4474
ARI: 0.11573257258598232
NMI: 0.11240087773587529


Actual Spam,0,1
Cluster,,
-1,4296,178
0,7,0
1,5,0
2,13,0
3,189,0
4,19,0
5,22,0
6,40,0
7,5,0


### Numerical + TF-IDF Features

In [19]:

import pandas as pd
import numpy as np
import time

from scipy.sparse import hstack, csr_matrix

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler, normalize
from sklearn.cluster import DBSCAN

from sklearn.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score
)

# ==========================================
# 1. SELECT REQUIRED COLUMNS
# ==========================================

selected_cols = [
    'is_spam',
    'url_count',
    'has_url',
    'word_count',
    'body_english'
]

clean_data_filtered = clean_data[selected_cols].copy()

# Separate features and target
X = clean_data_filtered.drop(columns=['is_spam'])
y = clean_data_filtered['is_spam']

# Handle missing values
X['body_english'] = (
    X['body_english'].fillna('').astype(str)
)

numeric_features = [
    'url_count',
    'has_url',
    'word_count'
]

X[numeric_features] = (
    X[numeric_features]
    .apply(pd.to_numeric, errors='coerce')
    .fillna(0)
)

# ==========================================
# 2. TRAIN-TEST SPLIT
# ==========================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

# ==========================================
# 3. TF-IDF VECTORIZATION
# ==========================================

start_time = time.time()

vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    dtype=np.float32
)

X_train_text = vectorizer.fit_transform(
    X_train['body_english']
)

X_test_text = vectorizer.transform(
    X_test['body_english']
)

print("\nTF-IDF completed in:",
      round(time.time() - start_time, 2), "seconds")

print("TF-IDF dimensions:", X_train_text.shape)

# ==========================================
# 4. EXTRACT NUMERICAL FEATURES
# ==========================================

X_train_num = X_train[numeric_features].values.astype(
    np.float32
)

X_test_num = X_test[numeric_features].values.astype(
    np.float32
)

# ==========================================
# 5. SCALE NUMERICAL FEATURES
# ==========================================

scaler = StandardScaler()

X_train_num_scaled = scaler.fit_transform(
    X_train_num
).astype(np.float32)

X_test_num_scaled = scaler.transform(
    X_test_num
).astype(np.float32)

# ==========================================
# 6. COMBINE NUMERICAL + TF-IDF
# ==========================================

X_train_combined = hstack([
    csr_matrix(X_train_num_scaled),
    X_train_text
], format='csr', dtype=np.float32)

X_test_combined = hstack([
    csr_matrix(X_test_num_scaled),
    X_test_text
], format='csr', dtype=np.float32)

print("\nCombined training dimensions:",
      X_train_combined.shape)

print("Combined testing dimensions:",
      X_test_combined.shape)

# ==========================================
# 7. NORMALIZE COMBINED FEATURES
# ==========================================

X_train_combined = normalize(
    X_train_combined,
    norm='l2'
)

X_test_combined = normalize(
    X_test_combined,
    norm='l2'
)

# ==========================================
# 8. DBSCAN EXPERIMENT
# ==========================================

# DBSCAN can be expensive on large datasets.
# Start with a reproducible sample of training data.

sample_size = min(5000, X_train_combined.shape[0])

rng = np.random.RandomState(42)

indices = rng.choice(
    X_train_combined.shape[0],
    size=sample_size,
    replace=False
)

X_sample = X_train_combined[indices]
y_sample = y_train.iloc[indices]

start_time = time.time()

dbscan = DBSCAN(
    eps=0.4,
    min_samples=5,
    metric='cosine',
    n_jobs=-1
)

clusters = dbscan.fit_predict(X_sample)

print("\nDBSCAN completed in:",
      round(time.time() - start_time, 2), "seconds")

# ==========================================
# 9. CLUSTERING EVALUATION
# ==========================================

n_clusters = len(set(clusters)) - (
    1 if -1 in clusters else 0
)

noise_count = np.sum(clusters == -1)

ari = adjusted_rand_score(
    y_sample,
    clusters
)

nmi = normalized_mutual_info_score(
    y_sample,
    clusters
)

print("\n========== DBSCAN RESULTS ==========")

print("Number of clusters:", n_clusters)
print("Noise points:", noise_count)

print(f"ARI: {ari:.4f}")
print(f"NMI: {nmi:.4f}")

# ==========================================
# 10. CLUSTER DISTRIBUTION
# ==========================================

results = pd.DataFrame({
    'Actual Spam': y_sample.values,
    'Cluster': clusters
})

cluster_distribution = pd.crosstab(
    results['Cluster'],
    results['Actual Spam'],
    rownames=['Cluster'],
    colnames=['Actual Class']
)

print("\n========== CLUSTER DISTRIBUTION ==========")

display(cluster_distribution)

# ==========================================
# 11. SPAM PERCENTAGE BY CLUSTER
# ==========================================

cluster_summary = results.groupby('Cluster')[
    'Actual Spam'
].agg(
    Total_Comments='count',
    Spam_Comments='sum'
)

cluster_summary['Non_Spam_Comments'] = (
    cluster_summary['Total_Comments']
    - cluster_summary['Spam_Comments']
)

cluster_summary['Spam_Percentage'] = (
    cluster_summary['Spam_Comments']
    / cluster_summary['Total_Comments'] * 100
)

print("\n========== SPAM DISTRIBUTION ==========")

display(
    cluster_summary.sort_values(
        'Spam_Percentage',
        ascending=False
    ).style.format({
        'Spam_Percentage': '{:.2f}%'
    })
)

# ==========================================
# 12. FINAL RESULTS
# ==========================================

final_results = pd.DataFrame([{
    'Algorithm': 'DBSCAN',
    'Features': 'Numerical + TF-IDF',
    'Clusters': n_clusters,
    'Noise Points': noise_count,
    'ARI': ari,
    'NMI': nmi
}])

print("\n========== FINAL PERFORMANCE ==========")

display(
    final_results.style.format({
        'ARI': '{:.4f}',
        'NMI': '{:.4f}'
    }).hide(axis='index')
)


Training samples: 74653
Testing samples: 18664

TF-IDF completed in: 17.23 seconds
TF-IDF dimensions: (74653, 5000)

Combined training dimensions: (74653, 5003)
Combined testing dimensions: (18664, 5003)

DBSCAN completed in: 0.76 seconds

========== DBSCAN RESULTS ==========
Number of clusters: 4
Noise points: 2606
ARI: 0.0342
NMI: 0.0443

========== CLUSTER DISTRIBUTION ==========


Actual Class,0,1
Cluster,,
-1,2575,31
0,1276,176
1,891,35
2,0,7
3,9,0



========== SPAM DISTRIBUTION ==========


,Total_Comments,Spam_Comments,Non_Spam_Comments,Spam_Percentage
Cluster,,,,
2,7,7,0,100.00%
0,1452,176,1276,12.12%
1,926,35,891,3.78%
-1,2606,31,2575,1.19%
3,9,0,9,0.00%



========== FINAL PERFORMANCE ==========


Algorithm,Features,Clusters,Noise Points,ARI,NMI
DBSCAN,Numerical + TF-IDF,4,2606,0.0342,0.0443


### Compute Embeddings

In [18]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, average_precision_score
from scipy.sparse import hstack, csr_matrix
from sentence_transformers import SentenceTransformer
from xgboost import XGBClassifier
import pandas as pd

# 1. Define the exact columns you want to select
selected_cols = ['is_spam', 'url_count', 'has_url', 'word_count', 'body_english']
clean_data_filtered = clean_data[selected_cols]

# 2. Separate into features (X) and target prediction (y)
X = clean_data_filtered.drop(columns=['is_spam']) 
y = clean_data_filtered['is_spam'] 

# FIX: Replace NaN values with an empty string and ensure everything is a string
X['body_english'] = X['body_english'].fillna('').astype(str)

# 3. Split data into training (80%) and testing (20%) sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 4. Apply TF-IDF to the text body
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_train_text = vectorizer.fit_transform(X_train['body_english'])
X_test_text = vectorizer.transform(X_test['body_english'])

# 5. Generate Text Embeddings using a lightweight, fast transformer model
print("Generating text embeddings...")
embedder = SentenceTransformer('all-MiniLM-L6-v2')
X_train_emb = embedder.encode(X_train['body_english'].tolist(), show_progress_bar=True)
X_test_emb = embedder.encode(X_test['body_english'].tolist(), show_progress_bar=True)

# 6. Extract numerical features
X_train_num = X_train[['url_count', 'has_url', 'word_count']].values
X_test_num = X_test[['url_count', 'has_url', 'word_count']].values

# 7. Combine numerical, TF-IDF, and embedding features into a single sparse matrix
# We convert the dense embedding arrays to sparse matrices for memory efficiency
X_train_combined = hstack([
    X_train_num, 
    X_train_text, 
    csr_matrix(X_train_emb)
])
X_test_combined = hstack([
    X_test_num, 
    X_test_text, 
    csr_matrix(X_test_emb)
])



Generating text embeddings...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/2333 [00:00<?, ?it/s]

Batches:   0%|          | 0/584 [00:00<?, ?it/s]

### Only Embeddings

In [20]:
# ==========================================
# 4. NORMALIZE EMBEDDINGS
# ==========================================

X_train_combined = normalize(
    X_train_emb,
    norm='l2'
)

X_test_combined = normalize(
    X_test_emb,
    norm='l2'
)

# ==========================================
# 5. DBSCAN CLUSTERING
# ==========================================

# Use a reproducible sample if dataset is large

sample_size = min(5000, len(X_train_combined))

rng = np.random.RandomState(42)

indices = rng.choice(
    len(X_train_combined),
    size=sample_size,
    replace=False
)

X_sample = X_train_combined[indices]
y_sample = y_train.iloc[indices]

start_time = time.time()

dbscan = DBSCAN(
    eps=0.3,
    min_samples=5,
    metric='cosine',
    n_jobs=-1
)

clusters = dbscan.fit_predict(X_sample)

print("\nDBSCAN completed in:",
      round(time.time() - start_time, 2), "seconds")

# ==========================================
# 6. CLUSTERING EVALUATION
# ==========================================

n_clusters = len(set(clusters)) - (
    1 if -1 in clusters else 0
)

noise_count = np.sum(clusters == -1)

ari = adjusted_rand_score(
    y_sample,
    clusters
)

nmi = normalized_mutual_info_score(
    y_sample,
    clusters
)

print("\n========== DBSCAN RESULTS ==========")

print("Number of clusters:", n_clusters)
print("Noise points:", noise_count)

print(f"ARI: {ari:.4f}")
print(f"NMI: {nmi:.4f}")

# ==========================================
# 7. CLUSTER DISTRIBUTION
# ==========================================

results = pd.DataFrame({
    'Actual Spam': y_sample.values,
    'Cluster': clusters
})

cluster_distribution = pd.crosstab(
    results['Cluster'],
    results['Actual Spam'],
    rownames=['Cluster'],
    colnames=['Actual Class']
)

print("\n========== CLUSTER DISTRIBUTION ==========")

display(cluster_distribution)

# ==========================================
# 8. SPAM PERCENTAGE BY CLUSTER
# ==========================================

cluster_summary = results.groupby('Cluster')[
    'Actual Spam'
].agg(
    Total_Comments='count',
    Spam_Comments='sum'
)

cluster_summary['Non_Spam_Comments'] = (
    cluster_summary['Total_Comments']
    - cluster_summary['Spam_Comments']
)

cluster_summary['Spam_Percentage'] = (
    cluster_summary['Spam_Comments']
    / cluster_summary['Total_Comments'] * 100
)

print("\n========== SPAM DISTRIBUTION ==========")

display(
    cluster_summary.sort_values(
        'Spam_Percentage',
        ascending=False
    ).style.format({
        'Spam_Percentage': '{:.2f}%'
    })
)

# ==========================================
# 9. FINAL RESULTS
# ==========================================

final_results = pd.DataFrame([{
    'Algorithm': 'DBSCAN',
    'Features': 'Sentence Embeddings',
    'Clusters': n_clusters,
    'Noise Points': noise_count,
    'ARI': ari,
    'NMI': nmi
}])

print("\n========== FINAL PERFORMANCE ==========")

display(
    final_results.style.format({
        'ARI': '{:.4f}',
        'NMI': '{:.4f}'
    }).hide(axis='index')
)


DBSCAN completed in: 0.68 seconds

========== DBSCAN RESULTS ==========
Number of clusters: 12
Noise points: 4108
ARI: -0.0090
NMI: 0.0717

========== CLUSTER DISTRIBUTION ==========


Actual Class,0,1
Cluster,,
-1,3890,218
0,802,0
1,5,0
2,8,0
3,7,0
4,12,0
5,0,6
6,0,13
7,17,0



========== SPAM DISTRIBUTION ==========


,Total_Comments,Spam_Comments,Non_Spam_Comments,Spam_Percentage
Cluster,,,,
5,6,6,0,100.00%
9,7,7,0,100.00%
11,5,5,0,100.00%
6,13,13,0,100.00%
-1,4108,218,3890,5.31%
3,7,0,7,0.00%
2,8,0,8,0.00%
1,5,0,5,0.00%
0,802,0,802,0.00%



========== FINAL PERFORMANCE ==========


Algorithm,Features,Clusters,Noise Points,ARI,NMI
DBSCAN,Sentence Embeddings,12,4108,-0.0090,0.0717


In [21]:

# ==========================================
# DBSCAN EPSILON EXPERIMENT
# ==========================================

eps_values = [0.2, 0.3, 0.4, 0.5, 0.6, 0.7]

experiment_results = []

for eps in eps_values:

    start_time = time.time()

    dbscan = DBSCAN(
        eps=eps,
        min_samples=5,
        metric='cosine',
        n_jobs=-1
    )

    clusters = dbscan.fit_predict(X_sample)

    n_clusters = len(set(clusters)) - (
        1 if -1 in clusters else 0
    )

    noise_count = np.sum(clusters == -1)

    noise_percentage = (
        noise_count / len(clusters)
    ) * 100

    ari = adjusted_rand_score(
        y_sample,
        clusters
    )

    nmi = normalized_mutual_info_score(
        y_sample,
        clusters
    )

    experiment_results.append({
        'Epsilon': eps,
        'Min Samples': 5,
        'Clusters': n_clusters,
        'Noise Points': noise_count,
        'Noise Percentage': noise_percentage,
        'ARI': ari,
        'NMI': nmi,
        'Time': round(time.time() - start_time, 2)
    })

eps_results = pd.DataFrame(experiment_results)

display(
    eps_results.style.format({
        'Noise Percentage': '{:.2f}%',
        'ARI': '{:.4f}',
        'NMI': '{:.4f}'
    })
)


,Epsilon,Min Samples,Clusters,Noise Points,Noise Percentage,ARI,NMI,Time
0,0.200000,5,9,4698,93.96%,-0.0208,0.0317,0.300000
1,0.300000,5,12,4108,82.16%,-0.0090,0.0717,0.220000
2,0.400000,5,8,2439,48.78%,0.0189,0.0508,0.260000
3,0.500000,5,6,756,15.12%,0.2088,0.1187,0.290000
4,0.600000,5,2,113,2.26%,0.3396,0.1821,0.310000
5,0.700000,5,1,14,0.28%,0.0602,0.0337,0.340000


In [23]:

# ==========================================
# BEST DBSCAN MODEL
# ==========================================

best_dbscan = DBSCAN(
    eps=0.6,
    min_samples=5,
    metric='cosine',
    n_jobs=-1
)

best_clusters = best_dbscan.fit_predict(X_sample)

clustered_comments = pd.DataFrame({
    'Comment': X_sample.shape[0] * [''],
    'Actual_Spam': y_sample.values,
    'Cluster': best_clusters
})

# Restore original comments
clustered_comments['Comment'] = (
    X_train.iloc[indices]['body_english'].values
)

# Show spam distribution
print(pd.crosstab(
    clustered_comments['Cluster'],
    clustered_comments['Actual_Spam']
))

# Examine comments in each cluster
for cluster_id in sorted(set(best_clusters)):

    print(f"\n========== CLUSTER {cluster_id} ==========")

    comments = clustered_comments[
        clustered_comments['Cluster'] == cluster_id
    ]['Comment']

    for comment in comments.head(10):
        print("-", comment)


Actual_Spam     0    1
Cluster               
-1             55   58
 0           4696  178
 1              0   13

========== CLUSTER -1 ==========
- There's a typo on “asking proving questions”
- Navigating Professionalism: Telemedicine's Impact on Long-Distance Medical Transport
Embarking on the journey of discussing the role of telemedicine in long-distance medical transport, it's essential to uphold the community guidelines and delve into a topic that not only aligns with our shared interests but also contributes positively to our collective knowledge.

Benefits of Virtual Consultations during Transit:

Timely Support: Telemedicine, https://medic-trans.com/non-emergency-medical-transport/ seamlessly integrated into long-distance medical transport, offers real-time communication channels, ensuring prompt access to medical advice. This timely support can be critical in addressing emergent situations during transit.

Collaborative Healthcare: The incorporation of telemedicine fosters

In [24]:

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    confusion_matrix,
    classification_report
)

# ==========================================
# EVALUATE CLUSTER 1 AS SPAM
# ==========================================

# Treat cluster 1 as predicted spam
y_pred_cluster = (best_clusters == 1).astype(int)

# Actual labels
y_true = y_sample.astype(int).values

print("========== CLUSTER-BASED SPAM EVALUATION ==========")

print("Accuracy:",
      accuracy_score(y_true, y_pred_cluster))

print("Precision:",
      precision_score(y_true, y_pred_cluster, zero_division=0))

print("Recall:",
      recall_score(y_true, y_pred_cluster, zero_division=0))

print("F1-score:",
      f1_score(y_true, y_pred_cluster, zero_division=0))

print("\nConfusion Matrix:")

print(confusion_matrix(y_true, y_pred_cluster))

print("\nClassification Report:")

print(classification_report(
    y_true,
    y_pred_cluster,
    zero_division=0
))


========== CLUSTER-BASED SPAM EVALUATION ==========
Accuracy: 0.9528
Precision: 1.0
Recall: 0.05220883534136546
F1-score: 0.09923664122137404

Confusion Matrix:
[[4751    0]
 [ 236   13]]

Classification Report:
              precision    recall  f1-score   support

           0       0.95      1.00      0.98      4751
           1       1.00      0.05      0.10       249

    accuracy                           0.95      5000
   macro avg       0.98      0.53      0.54      5000
weighted avg       0.96      0.95      0.93      5000



In [27]:

import hdbscan
import pandas as pd
import numpy as np

from sklearn.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score
)

# ==========================================
# HDBSCAN PARAMETER EXPERIMENT
# ==========================================

min_cluster_sizes = [5, 10, 15, 20, 30]
min_samples_values = [1, 3, 5, 10]

hdbscan_results = []

for min_size in min_cluster_sizes:

    for min_samples in min_samples_values:

        clusterer = hdbscan.HDBSCAN(
            min_cluster_size=min_size,
            min_samples=min_samples,
            metric='euclidean',
            cluster_selection_method='eom'
        )

        clusters = clusterer.fit_predict(X_sample)

        n_clusters = len(set(clusters)) - (
            1 if -1 in clusters else 0
        )

        noise_count = np.sum(clusters == -1)

        ari = adjusted_rand_score(
            y_sample,
            clusters
        )

        nmi = normalized_mutual_info_score(
            y_sample,
            clusters
        )

        hdbscan_results.append({
            'Min_Cluster_Size': min_size,
            'Min_Samples': min_samples,
            'Clusters': n_clusters,
            'Noise_Points': noise_count,
            'Noise_Percentage': (
                noise_count / len(clusters) * 100
            ),
            'ARI': ari,
            'NMI': nmi
        })

hdbscan_experiments = pd.DataFrame(
    hdbscan_results
)

display(
    hdbscan_experiments.sort_values(
        'ARI',
        ascending=False
    ).style.format({
        'Noise_Percentage': '{:.2f}%',
        'ARI': '{:.4f}',
        'NMI': '{:.4f}'
    })
)


,Min_Cluster_Size,Min_Samples,Clusters,Noise_Points,Noise_Percentage,ARI,NMI
1,5,3,30,4649,92.98%,0.2184,0.1620
5,10,3,13,4642,92.84%,0.1753,0.1504
2,5,5,21,4704,94.08%,0.1472,0.1223
6,10,5,11,4718,94.36%,0.1312,0.1196
4,10,1,26,4443,88.86%,0.1138,0.1143
0,5,1,108,4122,82.44%,0.1134,0.1051
8,15,1,12,4559,91.18%,0.1101,0.1160
12,20,1,8,4625,92.50%,0.0844,0.1002
16,30,1,6,4595,91.90%,0.0766,0.1013
13,20,3,7,4724,94.48%,0.0732,0.0877


### Numerical + TF-IDF + Embeddings

In [33]:

# ==========================================
# WEIGHTED HYBRID FEATURES
# ==========================================

from scipy.sparse import hstack, csr_matrix
from sklearn.preprocessing import normalize, StandardScaler
from sklearn.cluster import DBSCAN

from sklearn.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score
)

# ==========================================
# 1. NORMALIZE EMBEDDINGS
# ==========================================

X_emb_norm = normalize(
    X_sample_emb,
    norm='l2'
)

# ==========================================
# 2. SCALE NUMERICAL FEATURES
# ==========================================

scaler = StandardScaler()

X_num_scaled = scaler.fit_transform(
    X_sample_num
)

X_num_norm = normalize(
    X_num_scaled,
    norm='l2'
)

# ==========================================
# 3. EXPERIMENT WITH FEATURE WEIGHTS
# ==========================================

weights = [0.01, 0.05, 0.1, 0.2, 0.5, 1.0]

weighted_results = []

for weight in weights:

    X_weighted = hstack([
        csr_matrix(X_emb_norm),
        csr_matrix(X_num_norm * weight)
    ], format='csr')

    X_weighted = normalize(
        X_weighted,
        norm='l2'
    )

    # ======================================
    # DBSCAN
    # ======================================

    dbscan = DBSCAN(
        eps=0.6,
        min_samples=5,
        metric='cosine',
        n_jobs=-1
    )

    clusters = dbscan.fit_predict(X_weighted)

    n_clusters = len(set(clusters)) - (
        1 if -1 in clusters else 0
    )

    noise_count = np.sum(clusters == -1)

    ari = adjusted_rand_score(
        y_sample,
        clusters
    )

    nmi = normalized_mutual_info_score(
        y_sample,
        clusters
    )

    weighted_results.append({
        'Numerical_Weight': weight,
        'Clusters': n_clusters,
        'Noise_Points': noise_count,
        'Noise_Percentage': (
            noise_count / len(clusters) * 100
        ),
        'ARI': ari,
        'NMI': nmi
    })

# ==========================================
# 4. DISPLAY RESULTS
# ==========================================

weighted_df = pd.DataFrame(weighted_results)

display(
    weighted_df.style.format({
        'Noise_Percentage': '{:.2f}%',
        'ARI': '{:.4f}',
        'NMI': '{:.4f}'
    })
)


,Numerical_Weight,Clusters,Noise_Points,Noise_Percentage,ARI,NMI
0,0.010000,2,114,2.28%,0.3438,0.1849
1,0.050000,2,117,2.34%,0.3616,0.1977
2,0.100000,2,120,2.40%,0.3899,0.2194
3,0.200000,2,113,2.26%,0.4099,0.2399
4,0.500000,1,15,0.30%,0.0814,0.0530
5,1.000000,1,0,0.00%,0.0000,0.0000


In [34]:

# ==========================================
# FINE-TUNE NUMERICAL FEATURE WEIGHT
# ==========================================

fine_weights = [
    0.10,
    0.125,
    0.15,
    0.175,
    0.20,
    0.225,
    0.25,
    0.275,
    0.30
]

fine_results = []

for weight in fine_weights:

    X_weighted = hstack([
        csr_matrix(X_emb_norm),
        csr_matrix(X_num_norm * weight)
    ], format='csr')

    X_weighted = normalize(
        X_weighted,
        norm='l2'
    )

    model = DBSCAN(
        eps=0.6,
        min_samples=5,
        metric='cosine',
        n_jobs=-1
    )

    clusters = model.fit_predict(X_weighted)

    n_clusters = len(set(clusters)) - (
        1 if -1 in clusters else 0
    )

    noise = np.sum(clusters == -1)

    fine_results.append({
        'Weight': weight,
        'Clusters': n_clusters,
        'Noise': noise,
        'Noise_Percentage': noise / len(clusters) * 100,
        'ARI': adjusted_rand_score(y_sample, clusters),
        'NMI': normalized_mutual_info_score(
            y_sample, clusters
        )
    })

fine_df = pd.DataFrame(fine_results)

display(
    fine_df.sort_values(
        'ARI',
        ascending=False
    ).style.format({
        'Noise_Percentage': '{:.2f}%',
        'ARI': '{:.4f}',
        'NMI': '{:.4f}'
    })
)


,Weight,Clusters,Noise,Noise_Percentage,ARI,NMI
4,0.200000,2,113,2.26%,0.4099,0.2399
2,0.150000,2,119,2.38%,0.4072,0.2345
5,0.225000,2,109,2.18%,0.4045,0.2372
1,0.125000,2,126,2.52%,0.3980,0.2237
3,0.175000,2,111,2.22%,0.3963,0.2290
0,0.100000,2,120,2.40%,0.3899,0.2194
7,0.275000,2,92,1.84%,0.3885,0.2341
6,0.250000,2,98,1.96%,0.3860,0.2276
8,0.300000,2,86,1.72%,0.3852,0.2359


In [35]:

# ==========================================
# WEIGHTED HYBRID DBSCAN
# ==========================================

embedding_weights = [0.9, 0.8, 0.7, 0.6, 0.5]
numerical_weights = [0.1, 0.2, 0.3, 0.4, 0.5]

weight_results = []

for emb_weight, num_weight in zip(
    embedding_weights,
    numerical_weights
):

    X_weighted = hstack([
        csr_matrix(X_emb_norm * emb_weight),
        csr_matrix(X_num_norm * num_weight)
    ], format='csr')

    X_weighted = normalize(
        X_weighted,
        norm='l2'
    )

    model = DBSCAN(
        eps=0.6,
        min_samples=5,
        metric='cosine',
        n_jobs=-1
    )

    clusters = model.fit_predict(X_weighted)

    n_clusters = len(set(clusters)) - (
        1 if -1 in clusters else 0
    )

    noise = np.sum(clusters == -1)

    ari = adjusted_rand_score(
        y_sample, clusters
    )

    nmi = normalized_mutual_info_score(
        y_sample, clusters
    )

    weight_results.append({
        'Embedding_Weight': emb_weight,
        'Numerical_Weight': num_weight,
        'Clusters': n_clusters,
        'Noise_Percentage': noise / len(clusters) * 100,
        'ARI': ari,
        'NMI': nmi
    })

weight_df = pd.DataFrame(weight_results)

display(
    weight_df.sort_values(
        'ARI',
        ascending=False
    ).style.format({
        'Noise_Percentage': '{:.2f}%',
        'ARI': '{:.4f}',
        'NMI': '{:.4f}'
    })
)


,Embedding_Weight,Numerical_Weight,Clusters,Noise_Percentage,ARI,NMI
0,0.900000,0.100000,2,2.50%,0.3993,0.2252
1,0.800000,0.200000,2,1.96%,0.3860,0.2276
2,0.700000,0.300000,1,0.68%,0.1722,0.1097
3,0.600000,0.400000,1,0.00%,0.0000,0.0000
4,0.500000,0.500000,1,0.00%,0.0000,0.0000


### Diff weights for Numerical, TF-IDF & Embeddings

In [36]:

import numpy as np
import pandas as pd
import time

from scipy.sparse import hstack, csr_matrix

from sklearn.preprocessing import StandardScaler, normalize
from sklearn.cluster import DBSCAN
from sklearn.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score
)

# ==========================================
# 1. EXTRACT NUMERICAL FEATURES
# ==========================================

numeric_features = [
    'url_count',
    'has_url',
    'word_count'
]

X_train_num = (
    X_train[numeric_features]
    .apply(pd.to_numeric, errors='coerce')
    .fillna(0)
    .values.astype(np.float32)
)

# ==========================================
# 2. SELECT THE SAME 5000 SAMPLES
# ==========================================

X_sample_num = X_train_num[indices]

X_sample_tfidf = X_train_text[indices]

X_sample_emb = X_train_emb[indices]

y_sample = y_train.iloc[indices]

print("Numerical dimensions:", X_sample_num.shape)
print("TF-IDF dimensions:", X_sample_tfidf.shape)
print("Embedding dimensions:", X_sample_emb.shape)

# ==========================================
# 3. NORMALIZE NUMERICAL FEATURES
# ==========================================

scaler = StandardScaler()

X_num_scaled = scaler.fit_transform(
    X_sample_num
).astype(np.float32)

X_num_norm = normalize(
    X_num_scaled,
    norm='l2'
)

# ==========================================
# 4. NORMALIZE TF-IDF
# ==========================================

X_tfidf_norm = normalize(
    X_sample_tfidf,
    norm='l2'
)

# ==========================================
# 5. NORMALIZE EMBEDDINGS
# ==========================================

X_emb_norm = normalize(
    X_sample_emb,
    norm='l2'
)

# ==========================================
# 6. EXPERIMENT WITH THREE FEATURE WEIGHTS
# ==========================================

weight_combinations = [

    (0.10, 0.20, 0.70),
    (0.10, 0.30, 0.60),
    (0.10, 0.40, 0.50),

    (0.20, 0.20, 0.60),
    (0.20, 0.30, 0.50),
    (0.20, 0.40, 0.40),

    (0.30, 0.20, 0.50),
    (0.30, 0.30, 0.40),

    (0.05, 0.25, 0.70),
    (0.05, 0.35, 0.60)
]

weighted_results = []

# ==========================================
# 7. DBSCAN EXPERIMENT
# ==========================================

for num_weight, tfidf_weight, emb_weight in weight_combinations:

    start_time = time.time()

    # Combine three feature groups

    X_weighted = hstack([

        csr_matrix(
            X_num_norm * num_weight
        ),

        X_tfidf_norm * tfidf_weight,

        csr_matrix(
            X_emb_norm * emb_weight
        )

    ], format='csr', dtype=np.float32)

    # Final normalization

    X_weighted = normalize(
        X_weighted,
        norm='l2'
    )

    # DBSCAN

    dbscan = DBSCAN(
        eps=0.6,
        min_samples=5,
        metric='cosine',
        n_jobs=-1
    )

    clusters = dbscan.fit_predict(
        X_weighted
    )

    # Evaluation

    n_clusters = len(set(clusters)) - (
        1 if -1 in clusters else 0
    )

    noise_count = np.sum(clusters == -1)

    ari = adjusted_rand_score(
        y_sample,
        clusters
    )

    nmi = normalized_mutual_info_score(
        y_sample,
        clusters
    )

    elapsed_time = time.time() - start_time

    weighted_results.append({

        'Numerical_Weight': num_weight,

        'TFIDF_Weight': tfidf_weight,

        'Embedding_Weight': emb_weight,

        'Clusters': n_clusters,

        'Noise_Points': noise_count,

        'Noise_Percentage': (
            noise_count / len(clusters) * 100
        ),

        'ARI': ari,

        'NMI': nmi,

        'Time': elapsed_time

    })

    print(
        f"Completed: Numerical={num_weight}, "
        f"TF-IDF={tfidf_weight}, "
        f"Embedding={emb_weight}, "
        f"ARI={ari:.4f}, NMI={nmi:.4f}"
    )

# ==========================================
# 8. DISPLAY RESULTS
# ==========================================

weighted_df = pd.DataFrame(weighted_results)

weighted_df = weighted_df.sort_values(
    'ARI',
    ascending=False
)

print("\n========== THREE-FEATURE DBSCAN RESULTS ==========")

display(
    weighted_df.style.format({
        'Noise_Percentage': '{:.2f}%',
        'ARI': '{:.4f}',
        'NMI': '{:.4f}',
        'Time': '{:.2f}s'
    })
)


Numerical dimensions: (5000, 3)
TF-IDF dimensions: (5000, 5000)
Embedding dimensions: (5000, 384)
Completed: Numerical=0.1, TF-IDF=0.2, Embedding=0.7, ARI=0.4972, NMI=0.2906
Completed: Numerical=0.1, TF-IDF=0.3, Embedding=0.6, ARI=0.3841, NMI=0.2205
Completed: Numerical=0.1, TF-IDF=0.4, Embedding=0.5, ARI=0.0857, NMI=0.0758
Completed: Numerical=0.2, TF-IDF=0.2, Embedding=0.6, ARI=0.4700, NMI=0.2831
Completed: Numerical=0.2, TF-IDF=0.3, Embedding=0.5, ARI=0.4312, NMI=0.2345
Completed: Numerical=0.2, TF-IDF=0.4, Embedding=0.4, ARI=0.1139, NMI=0.0897
Completed: Numerical=0.3, TF-IDF=0.2, Embedding=0.5, ARI=0.0626, NMI=0.0451
Completed: Numerical=0.3, TF-IDF=0.3, Embedding=0.4, ARI=0.0887, NMI=0.0614
Completed: Numerical=0.05, TF-IDF=0.25, Embedding=0.7, ARI=0.4853, NMI=0.2830
Completed: Numerical=0.05, TF-IDF=0.35, Embedding=0.6, ARI=0.2495, NMI=0.1490

========== THREE-FEATURE DBSCAN RESULTS ==========


,Numerical_Weight,TFIDF_Weight,Embedding_Weight,Clusters,Noise_Points,Noise_Percentage,ARI,NMI,Time
0,0.100000,0.200000,0.700000,5,181,3.62%,0.4972,0.2906,5.47s
8,0.050000,0.250000,0.700000,6,237,4.74%,0.4853,0.2830,14.38s
3,0.200000,0.200000,0.600000,4,119,2.38%,0.4700,0.2831,4.83s
4,0.200000,0.300000,0.500000,5,227,4.54%,0.4312,0.2345,4.82s
1,0.100000,0.300000,0.600000,6,377,7.54%,0.3841,0.2205,5.17s
9,0.050000,0.350000,0.600000,9,602,12.04%,0.2495,0.1490,25.30s
5,0.200000,0.400000,0.400000,9,1079,21.58%,0.1139,0.0897,9.90s
7,0.300000,0.300000,0.400000,1,15,0.30%,0.0887,0.0614,17.27s
2,0.100000,0.400000,0.500000,12,1355,27.10%,0.0857,0.0758,4.86s
6,0.300000,0.200000,0.500000,1,10,0.20%,0.0626,0.0451,20.89s


In [37]:
# Rebuild the best configuration
num_weight = 0.10
tfidf_weight = 0.20
emb_weight = 0.70

X_best = hstack([
    csr_matrix(X_num_norm * num_weight),
    X_tfidf_norm * tfidf_weight,
    csr_matrix(X_emb_norm * emb_weight)
], format='csr')

X_best = normalize(X_best, norm='l2')

# Run DBSCAN
best_dbscan = DBSCAN(
    eps=0.6,
    min_samples=5,
    metric='cosine',
    n_jobs=-1
)

best_clusters = best_dbscan.fit_predict(X_best)

# Create evaluation dataframe
results = pd.DataFrame({
    'Actual_Spam': y_sample.values,
    'Cluster': best_clusters
})

# Calculate cluster statistics
summary = results.groupby('Cluster')['Actual_Spam'].agg(
    Total_Comments='count',
    Spam_Comments='sum'
)

summary['Non_Spam_Comments'] = (
    summary['Total_Comments'] - summary['Spam_Comments']
)

summary['Spam_Percentage'] = (
    summary['Spam_Comments'] /
    summary['Total_Comments'] * 100
)

total_spam = results['Actual_Spam'].sum()

summary['Spam_Capture_Rate'] = (
    summary['Spam_Comments'] / total_spam * 100
)

summary['Cluster_Purity'] = (
    summary[['Spam_Comments', 'Non_Spam_Comments']]
    .max(axis=1) /
    summary['Total_Comments'] * 100
)

display(
    summary.sort_values(
        'Spam_Percentage',
        ascending=False
    ).style.format({
        'Spam_Percentage': '{:.2f}%',
        'Spam_Capture_Rate': '{:.2f}%',
        'Cluster_Purity': '{:.2f}%'
    })
)

,Total_Comments,Spam_Comments,Non_Spam_Comments,Spam_Percentage,Spam_Capture_Rate,Cluster_Purity
Cluster,,,,,,
2,14,14,0,100.00%,5.62%,100.00%
1,5,5,0,100.00%,2.01%,100.00%
3,13,13,0,100.00%,5.22%,100.00%
4,6,6,0,100.00%,2.41%,100.00%
-1,181,89,92,49.17%,35.74%,50.83%
0,4781,122,4659,2.55%,49.00%,97.45%


In [38]:

from sklearn.neighbors import NearestNeighbors
from sklearn.cluster import DBSCAN
from scipy.sparse import csr_matrix
from scipy.sparse import hstack
from sklearn.preprocessing import normalize

import numpy as np
import pandas as pd
import time

# ==========================================
# 1. PREPARE ALL 91K FEATURES
# ==========================================

# Assuming these contain all training observations:

# X_train_num
# X_train_text
# X_train_emb

# Scale numerical features
scaler = StandardScaler()

X_num_scaled = scaler.fit_transform(
    X_train_num
).astype(np.float32)

X_num_norm = normalize(X_num_scaled)

# Normalize TF-IDF
X_tfidf_norm = normalize(X_train_text)

# Normalize embeddings
X_emb_norm = normalize(X_train_emb)

# ==========================================
# 2. COMBINE THREE FEATURE GROUPS
# ==========================================

num_weight = 0.10
tfidf_weight = 0.20
emb_weight = 0.70

X_combined = hstack([

    csr_matrix(X_num_norm * num_weight),

    X_tfidf_norm * tfidf_weight,

    csr_matrix(X_emb_norm * emb_weight)

], format='csr', dtype=np.float32)

X_combined = normalize(X_combined, norm='l2')

print("Full dataset dimensions:", X_combined.shape)

# ==========================================
# 3. BUILD NEIGHBOUR GRAPH
# ==========================================

start_time = time.time()

eps = 0.6

neighbors = NearestNeighbors(
    radius=eps,
    metric='cosine',
    algorithm='brute',
    n_jobs=-1
)

neighbors.fit(X_combined)

# Sparse distance graph
graph = neighbors.radius_neighbors_graph(
    X_combined,
    mode='distance'
)

print(
    "Neighbour graph completed:",
    round(time.time() - start_time, 2),
    "seconds"
)

print("Graph shape:", graph.shape)
print("Non-zero distances:", graph.nnz)

# ==========================================
# 4. RUN DBSCAN USING PRECOMPUTED DISTANCES
# ==========================================

start_time = time.time()

dbscan = DBSCAN(
    eps=eps,
    min_samples=5,
    metric='precomputed',
    n_jobs=-1
)

clusters = dbscan.fit_predict(graph)

print(
    "DBSCAN completed:",
    round(time.time() - start_time, 2),
    "seconds"
)

# ==========================================
# 5. EVALUATE CLUSTERS
# ==========================================

n_clusters = len(set(clusters)) - (
    1 if -1 in clusters else 0
)

noise_count = np.sum(clusters == -1)

ari = adjusted_rand_score(
    y_train,
    clusters
)

nmi = normalized_mutual_info_score(
    y_train,
    clusters
)

print("\n========== FULL DATASET DBSCAN ==========")

print("Observations:", len(clusters))
print("Clusters:", n_clusters)
print("Noise points:", noise_count)

print(
    "Noise percentage:",
    round(noise_count / len(clusters) * 100, 2),
    "%"
)

print(f"ARI: {ari:.4f}")
print(f"NMI: {nmi:.4f}")


Full dataset dimensions: (74653, 5387)
Neighbour graph completed: 2791.14 seconds
Graph shape: (74653, 74653)
Non-zero distances: 131171627
DBSCAN completed: 38.98 seconds

========== FULL DATASET DBSCAN ==========
Observations: 74653
Clusters: 3
Noise points: 369
Noise percentage: 0.49 %
ARI: 0.1282
NMI: 0.0773


In [39]:
# ==========================================
# EPSILON OPTIMISATION
# ==========================================

eps_values = [0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70]

eps_results = []

for eps in eps_values:

    start_time = time.time()

    dbscan = DBSCAN(
        eps=eps,
        min_samples=5,
        metric='precomputed',
        n_jobs=-1
    )

    clusters_test = dbscan.fit_predict(graph)

    n_clusters = len(set(clusters_test)) - (
        1 if -1 in clusters_test else 0
    )

    noise = np.sum(clusters_test == -1)

    ari = adjusted_rand_score(
        y_train,
        clusters_test
    )

    nmi = normalized_mutual_info_score(
        y_train,
        clusters_test
    )

    eps_results.append({
        'Epsilon': eps,
        'Clusters': n_clusters,
        'Noise_Points': noise,
        'Noise_Percentage': noise / len(clusters_test) * 100,
        'ARI': ari,
        'NMI': nmi,
        'Time': time.time() - start_time
    })

eps_df = pd.DataFrame(eps_results)

display(
    eps_df.style.format({
        'Noise_Percentage': '{:.2f}%',
        'ARI': '{:.4f}',
        'NMI': '{:.4f}',
        'Time': '{:.2f}s'
    })
)

,Epsilon,Clusters,Noise_Points,Noise_Percentage,ARI,NMI,Time
0,0.400000,115,16107,21.58%,0.1296,0.1091,25.64s
1,0.450000,40,6378,8.54%,0.3008,0.1777,13.43s
2,0.500000,19,2377,3.18%,0.3275,0.1611,12.33s
3,0.550000,10,962,1.29%,0.2609,0.1439,11.88s
4,0.600000,3,369,0.49%,0.1282,0.0773,10.07s
5,0.650000,3,369,0.49%,0.1282,0.0773,11.56s
6,0.700000,3,369,0.49%,0.1282,0.0773,11.73s


In [41]:

# ============================================================
# HYBRID UNSUPERVISED SPAM DETECTION
# Numerical + TF-IDF + Sentence Embeddings + DBSCAN
# ============================================================

import pandas as pd
import numpy as np
import time

from scipy.sparse import hstack, csr_matrix

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.preprocessing import StandardScaler, normalize

from sklearn.cluster import DBSCAN
from sklearn.neighbors import NearestNeighbors

from sklearn.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score
)

from sentence_transformers import SentenceTransformer


# ============================================================
# 1. SELECT REQUIRED COLUMNS
# ============================================================

selected_cols = [
    'is_spam',
    'url_count',
    'has_url',
    'word_count',
    'body_english'
]

clean_data_filtered = clean_data[selected_cols].copy()

# Separate features and target
X = clean_data_filtered.drop(columns=['is_spam'])
y = clean_data_filtered['is_spam'].astype(int)

# Handle missing text
X['body_english'] = (
    X['body_english']
    .fillna('')
    .astype(str)
)

# Handle missing numerical values
numeric_features = [
    'url_count',
    'has_url',
    'word_count'
]

X[numeric_features] = (
    X[numeric_features]
    .apply(pd.to_numeric, errors='coerce')
    .fillna(0)
)

print("Total observations:", len(X))


# ============================================================
# 5. EXTRACT NUMERICAL FEATURES
# ============================================================

X_train_num = X_train[numeric_features].values.astype(
    np.float32
)

X_test_num = X_test[numeric_features].values.astype(
    np.float32
)

print("\nNumerical dimensions:", X_train_num.shape)


# ============================================================
# 6. SCALE AND NORMALIZE NUMERICAL FEATURES
# ============================================================

scaler = StandardScaler()

X_train_num_scaled = scaler.fit_transform(
    X_train_num
).astype(np.float32)

X_test_num_scaled = scaler.transform(
    X_test_num
).astype(np.float32)

X_train_num_norm = normalize(
    X_train_num_scaled,
    norm='l2'
)

X_test_num_norm = normalize(
    X_test_num_scaled,
    norm='l2'
)


# ============================================================
# 7. NORMALIZE TF-IDF AND EMBEDDINGS
# ============================================================

X_train_tfidf_norm = normalize(
    X_train_text,
    norm='l2'
)

X_test_tfidf_norm = normalize(
    X_test_text,
    norm='l2'
)

X_train_emb_norm = normalize(
    X_train_emb,
    norm='l2'
)

X_test_emb_norm = normalize(
    X_test_emb,
    norm='l2'
)


# ============================================================
# 8. APPLY FEATURE WEIGHTS
# ============================================================

# Best configuration from your experiments

num_weight = 0.10
tfidf_weight = 0.20
emb_weight = 0.70

print("\n========== FEATURE WEIGHTS ==========")

print("Numerical weight:", num_weight)
print("TF-IDF weight:", tfidf_weight)
print("Embedding weight:", emb_weight)

print(
    "Total weight:",
    num_weight + tfidf_weight + emb_weight
)


# ============================================================
# 9. COMBINE THREE FEATURE GROUPS
# ============================================================

X_train_combined = hstack([

    csr_matrix(
        X_train_num_norm * num_weight
    ),

    X_train_tfidf_norm * tfidf_weight,

    csr_matrix(
        X_train_emb_norm * emb_weight
    )

], format='csr', dtype=np.float32)


X_test_combined = hstack([

    csr_matrix(
        X_test_num_norm * num_weight
    ),

    X_test_tfidf_norm * tfidf_weight,

    csr_matrix(
        X_test_emb_norm * emb_weight
    )

], format='csr', dtype=np.float32)


# Final L2 normalization
X_train_combined = normalize(
    X_train_combined,
    norm='l2'
)

X_test_combined = normalize(
    X_test_combined,
    norm='l2'
)

print("\n========== HYBRID FEATURES ==========")

print("Training dimensions:", X_train_combined.shape)
print("Testing dimensions:", X_test_combined.shape)


# ============================================================
# 10. BUILD TRAINING NEIGHBOUR GRAPH
# ============================================================

start_time = time.time()

eps = 0.50
min_samples = 5

neighbors_graph_model = NearestNeighbors(
    radius=eps,
    metric='cosine',
    algorithm='brute',
    n_jobs=-1
)

neighbors_graph_model.fit(X_train_combined)

train_graph = neighbors_graph_model.radius_neighbors_graph(
    X_train_combined,
    mode='distance'
)

# Ensure traditional CSR sparse matrix format
train_graph = csr_matrix(train_graph, dtype=np.float64)

# Explicitly set the diagonal to zero
train_graph.setdiag(0)

train_graph.eliminate_zeros()

print(
    "\nNeighbour graph completed:",
    round(time.time() - start_time, 2),
    "seconds"
)

print("Graph shape:", train_graph.shape)
print("Non-zero distances:", train_graph.nnz)


# ============================================================
# 11. FIT DBSCAN ON TRAINING DATA
# ============================================================

start_time = time.time()

dbscan = DBSCAN(
    eps=eps,
    min_samples=min_samples,
    metric='precomputed',
    n_jobs=-1
)

train_clusters = dbscan.fit_predict(
    train_graph
)

print("\nDBSCAN completed in:",
      round(time.time() - start_time, 2),
      "seconds")


# ============================================================
# 12. TRAINING CLUSTER EVALUATION
# ============================================================

n_clusters = len(set(train_clusters)) - (
    1 if -1 in train_clusters else 0
)

noise_count = np.sum(train_clusters == -1)

noise_percentage = (
    noise_count / len(train_clusters) * 100
)

ari_train = adjusted_rand_score(
    y_train,
    train_clusters
)

nmi_train = normalized_mutual_info_score(
    y_train,
    train_clusters
)

print("\n========== TRAINING DBSCAN RESULTS ==========")

print("Observations:", len(train_clusters))
print("Clusters:", n_clusters)
print("Noise points:", noise_count)

print(f"Noise percentage: {noise_percentage:.2f}%")

print(f"ARI: {ari_train:.4f}")
print(f"NMI: {nmi_train:.4f}")


# ============================================================
# 13. TRAINING CLUSTER DISTRIBUTION
# ============================================================

train_results = pd.DataFrame({
    'Actual_Spam': y_train.to_numpy(),
    'Cluster': train_clusters
})

train_distribution = pd.crosstab(
    train_results['Cluster'],
    train_results['Actual_Spam'],
    rownames=['Cluster'],
    colnames=['Actual Class']
)

print("\n========== TRAINING CLUSTER DISTRIBUTION ==========")

display(train_distribution)


# ============================================================
# 14. TRAINING SPAM PERCENTAGE
# ============================================================

train_summary = train_results.groupby('Cluster')[
    'Actual_Spam'
].agg(
    Total_Comments='count',
    Spam_Comments='sum'
)

train_summary['Non_Spam_Comments'] = (
    train_summary['Total_Comments']
    - train_summary['Spam_Comments']
)

train_summary['Spam_Percentage'] = (
    train_summary['Spam_Comments']
    / train_summary['Total_Comments'] * 100
)

total_train_spam = train_results['Actual_Spam'].sum()

train_summary['Spam_Capture_Rate'] = (
    train_summary['Spam_Comments']
    / total_train_spam * 100
)

train_summary['Cluster_Purity'] = (
    train_summary[
        ['Spam_Comments', 'Non_Spam_Comments']
    ].max(axis=1)
    / train_summary['Total_Comments'] * 100
)

print("\n========== TRAINING SPAM DISTRIBUTION ==========")

display(
    train_summary.sort_values(
        'Spam_Percentage',
        ascending=False
    ).style.format({
        'Spam_Percentage': '{:.2f}%',
        'Spam_Capture_Rate': '{:.2f}%',
        'Cluster_Purity': '{:.2f}%'
    })
)


# ============================================================
# 15. LEARN CLUSTER-TO-SPAM MAPPING
# ============================================================

# Use training labels only.
# Majority voting determines the class of each cluster.

train_mapping = {}

for cluster_id in np.unique(train_clusters):

    # Exclude DBSCAN noise
    if cluster_id == -1:
        continue

    mask = train_clusters == cluster_id

    spam_count = y_train.to_numpy()[mask].sum()

    non_spam_count = mask.sum() - spam_count

    train_mapping[cluster_id] = int(
        spam_count > non_spam_count
    )

print("\n========== TRAINING CLUSTER MAPPING ==========")

print(train_mapping)


# ============================================================
# 16. ASSIGN TEST DATA TO TRAINING CLUSTERS
# ============================================================

# DBSCAN has no native predict() method.
# Use nearest-neighbour assignment as an approximation.

start_time = time.time()

nn = NearestNeighbors(
    radius=eps,
    metric='cosine',
    algorithm='brute',
    n_jobs=-1
)

nn.fit(X_train_combined)

test_distances, test_indices = nn.radius_neighbors(
    X_test_combined,
    return_distance=True
)

print(
    "\nTest neighbour search completed in:",
    round(time.time() - start_time, 2),
    "seconds"
)


# ============================================================
# 17. PREDICT TEST CLASSES
# ============================================================

y_pred_test = np.zeros(
    X_test_combined.shape[0],
    dtype=int
)

unassigned = np.zeros(
    X_test_combined.shape[0],
    dtype=bool
)

for i in range(X_test_combined.shape[0]):

    dist = test_distances[i]
    idx = test_indices[i]

    # Exclude training noise observations
    valid = train_clusters[idx] != -1

    dist = dist[valid]
    idx = idx[valid]

    if idx.size == 0:
        unassigned[i] = True
        continue

    neighbour_clusters = train_clusters[idx]

    cluster_votes = {}

    for cluster_id, distance in zip(
        neighbour_clusters,
        dist
    ):

        if cluster_id not in train_mapping:
            continue

        vote = 1.0 / max(distance, 1e-6)

        cluster_votes[cluster_id] = (
            cluster_votes.get(cluster_id, 0) + vote
        )

    if not cluster_votes:
        unassigned[i] = True
        continue

    best_cluster = max(
        cluster_votes,
        key=cluster_votes.get
    )

    y_pred_test[i] = train_mapping[best_cluster]


print("\n========== TEST ASSIGNMENT ==========")

print("Test observations:", X_test_combined.shape[0])

print("Assigned observations:", np.sum(~unassigned))

print("Unassigned observations:", np.sum(unassigned))

print(
    "Unassigned percentage:",
    round(unassigned.mean() * 100, 2),
    "%"
)


# ============================================================
# 18. TEST CLASSIFICATION EVALUATION
# ============================================================

# Unassigned observations are retained as non-spam
# for this explicit binary classification policy.
# Also report assigned-only results separately.

print("\n========== TEST RESULTS (ALL OBSERVATIONS) ==========")

print("\nConfusion Matrix:")

print(
    confusion_matrix(
        y_test,
        y_pred_test,
        labels=[0, 1]
    )
)

print("\nClassification Report:")

print(
    classification_report(
        y_test,
        y_pred_test,
        labels=[0, 1],
        zero_division=0
    )
)

precision = precision_score(
    y_test,
    y_pred_test,
    zero_division=0
)

recall = recall_score(
    y_test,
    y_pred_test,
    zero_division=0
)

f1 = f1_score(
    y_test,
    y_pred_test,
    zero_division=0
)

accuracy = accuracy_score(
    y_test,
    y_pred_test
)

print("Accuracy:", round(accuracy, 4))
print("Precision:", round(precision, 4))
print("Recall:", round(recall, 4))
print("F1-score:", round(f1, 4))


# ============================================================
# 19. ASSIGNED-ONLY TEST EVALUATION
# ============================================================

# Evaluate only observations assigned to a training cluster.
# This measures performance conditional on receiving an assignment.

assigned_mask = ~unassigned

if assigned_mask.sum() > 0:

    y_test_assigned = y_test.to_numpy()[assigned_mask]

    y_pred_assigned = y_pred_test[assigned_mask]

    print("\n========== ASSIGNED-ONLY TEST RESULTS ==========")

    print(
        classification_report(
            y_test_assigned,
            y_pred_assigned,
            labels=[0, 1],
            zero_division=0
        )
    )

    print(
        "Assigned-only precision:",
        round(
            precision_score(
                y_test_assigned,
                y_pred_assigned,
                zero_division=0
            ),
            4
        )
    )

    print(
        "Assigned-only recall:",
        round(
            recall_score(
                y_test_assigned,
                y_pred_assigned,
                zero_division=0
            ),
            4
        )
    )

    print(
        "Assigned-only F1:",
        round(
            f1_score(
                y_test_assigned,
                y_pred_assigned,
                zero_division=0
            ),
            4
        )
    )


# ============================================================
# 20. FINAL PERFORMANCE SUMMARY
# ============================================================

final_results = pd.DataFrame([{

    'Algorithm': 'Hybrid DBSCAN',

    'Features': 'Numerical + TF-IDF + Embeddings',

    'Numerical Weight': num_weight,

    'TF-IDF Weight': tfidf_weight,

    'Embedding Weight': emb_weight,

    'Epsilon': eps,

    'Min Samples': min_samples,

    'Training Clusters': n_clusters,

    'Training ARI': ari_train,

    'Training NMI': nmi_train,

    'Training Noise %': noise_percentage,

    'Test Accuracy': accuracy,

    'Test Precision': precision,

    'Test Recall': recall,

    'Test F1': f1,

    'Test Unassigned %': unassigned.mean() * 100

}])

print("\n========== FINAL PERFORMANCE SUMMARY ==========")

display(
    final_results.style.format({
        'Numerical Weight': '{:.2f}',
        'TF-IDF Weight': '{:.2f}',
        'Embedding Weight': '{:.2f}',
        'Epsilon': '{:.2f}',
        'Training ARI': '{:.4f}',
        'Training NMI': '{:.4f}',
        'Training Noise %': '{:.2f}%',
        'Test Accuracy': '{:.4f}',
        'Test Precision': '{:.4f}',
        'Test Recall': '{:.4f}',
        'Test F1': '{:.4f}',
        'Test Unassigned %': '{:.2f}%'
    }).hide(axis='index')
)


Total observations: 93317

Numerical dimensions: (74653, 3)

========== FEATURE WEIGHTS ==========
Numerical weight: 0.1
TF-IDF weight: 0.2
Embedding weight: 0.7
Total weight: 1.0

========== HYBRID FEATURES ==========
Training dimensions: (74653, 5387)
Testing dimensions: (18664, 5387)

Neighbour graph completed: 2005.77 seconds
Graph shape: (74653, 74653)
Non-zero distances: 30382660

DBSCAN completed in: 4.32 seconds

========== TRAINING DBSCAN RESULTS ==========
Observations: 74653
Clusters: 19
Noise points: 2377
Noise percentage: 3.18%
ARI: 0.3275
NMI: 0.1611

========== TRAINING CLUSTER DISTRIBUTION ==========


Actual Class,0,1
Cluster,,
-1,1482,895
0,69652,2363
1,0,133
2,5,0
3,0,19
4,0,5
5,0,14
6,0,8
7,0,7



========== TRAINING SPAM DISTRIBUTION ==========


,Total_Comments,Spam_Comments,Non_Spam_Comments,Spam_Percentage,Spam_Capture_Rate,Cluster_Purity
Cluster,,,,,,
1,133,133,0,100.00%,3.79%,100.00%
5,14,14,0,100.00%,0.40%,100.00%
4,5,5,0,100.00%,0.14%,100.00%
3,19,19,0,100.00%,0.54%,100.00%
11,5,5,0,100.00%,0.14%,100.00%
12,9,9,0,100.00%,0.26%,100.00%
13,5,5,0,100.00%,0.14%,100.00%
6,8,8,0,100.00%,0.23%,100.00%
7,7,7,0,100.00%,0.20%,100.00%



========== TRAINING CLUSTER MAPPING ==========
{np.int64(0): 0, np.int64(1): 1, np.int64(2): 0, np.int64(3): 1, np.int64(4): 1, np.int64(5): 1, np.int64(6): 1, np.int64(7): 1, np.int64(8): 1, np.int64(9): 1, np.int64(10): 1, np.int64(11): 1, np.int64(12): 1, np.int64(13): 1, np.int64(14): 1, np.int64(15): 1, np.int64(16): 0, np.int64(17): 1, np.int64(18): 0}

Test neighbour search completed in: 423.73 seconds

========== TEST ASSIGNMENT ==========
Test observations: 18664
Assigned observations: 18077
Unassigned observations: 587
Unassigned percentage: 3.15 %

========== TEST RESULTS (ALL OBSERVATIONS) ==========

Confusion Matrix:
[[17787     1]
 [  793    83]]

Classification Report:
              precision    recall  f1-score   support

           0       0.96      1.00      0.98     17788
           1       0.99      0.09      0.17       876

    accuracy                           0.96     18664
   macro avg       0.97      0.55      0.58     18664
weighted avg       0.96      0.96

Algorithm,Features,Numerical Weight,TF-IDF Weight,Embedding Weight,Epsilon,Min Samples,Training Clusters,Training ARI,Training NMI,Training Noise %,Test Accuracy,Test Precision,Test Recall,Test F1,Test Unassigned %
Hybrid DBSCAN,Numerical + TF-IDF + Embeddings,0.10,0.20,0.70,0.50,5,19,0.3275,0.1611,3.18%,0.9575,0.9881,0.0947,0.1729,3.15%
